In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

# 1. 데이터셋 로드
dataset = Planetoid(root="data/Planetoid", name="Cora")
data = dataset[0]

print(data)
print("노드 수:", data.num_nodes)
print("엣지 수:", data.num_edges)
print("feature 차원:", dataset.num_node_features)
print("클래스 수:", dataset.num_classes)


# 2. 모델 정의
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        # 1-hop 이웃 정보 반영
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        # 다시 한 번 이웃 정보 반영
        x = self.conv2(x, edge_index)

        # 각 노드가 각 클래스일 log-probability
        return F.log_softmax(x, dim=1)


# 3. 모델/옵티마이저 준비
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GCN(
    in_channels=dataset.num_node_features,
    hidden_channels=16,
    out_channels=dataset.num_classes,
).to(device)

data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)


# 4. 학습 함수
def train():
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index)

    # train_mask에 해당하는 노드만 loss 계산
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])

    loss.backward()
    optimizer.step()
    return loss.item()


# 5. 평가 함수
@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        correct = (pred[mask] == data.y[mask]).sum()
        acc = int(correct) / int(mask.sum())
        accs.append(acc)
    return accs


# 6. 학습 루프
for epoch in range(1, 201):
    loss = train()
    train_acc, val_acc, test_acc = test()

    if epoch % 20 == 0:
        print(
            f"Epoch: {epoch:03d}, "
            f"Loss: {loss:.4f}, "
            f"Train: {train_acc:.4f}, "
            f"Val: {val_acc:.4f}, "
            f"Test: {test_acc:.4f}"
        )

In [ ]:
# Visualization (선택 사항)
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE


@torch.no_grad()
def visualize():
    model.eval()
    out = model(data.x, data.edge_index)
    out = out.cpu().numpy()
    y = data.y.cpu().numpy()

    tsne = TSNE(n_components=2, random_state=42)
    out_2d = tsne.fit_transform(out)

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(out_2d[:, 0], out_2d[:, 1], c=y, cmap="tab10", s=50)
    plt.legend(
        handles=scatter.legend_elements()[0],
        labels=[f"Class {i}" for i in range(dataset.num_classes)],
        title="Classes",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
    )
    plt.title("t-SNE Visualization of Node Embeddings")
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.tight_layout()
    plt.show()


visualize()